# Notebook 00 · Sensor Preprocessing

This notebook handles raw data extraction specifically for on-site sensors:

1. **File Inspection** — Examine an unknown raw sensor file before loading
2. **Data Loading** — Read all raw sensor files from `/data/raw/sensor/`
3. **Signal Cleaning** — Remove power-loss rows and physical outliers (optional)
4. **Save** — Export the cleaned, standardized dataset to `/data/interim/sensor/`


In [1]:
import sys
import os
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))

from heritageshm.dataloader import inspect_raw_file, load_sensor_directory, organize_sensor_data
from heritageshm.preprocessing import clean_signal, apply_compensation, temp_compensation

## Step 1 · File Inspection
Before loading all files, inspect one representative raw file to identify the delimiter, 
column count, and date format.

In [6]:
# === USER INPUT ===
# Replace with the path to any one raw file in your data folder
sample_file = r"./data/raw/sensor/GUBBIO_20180726.adc"
# ==================

inspect_raw_file(sample_file)

--- Inspecting File: GUBBIO_20180726.adc ---
Detected Delimiter: ','
Detected Columns: 13

First 5 lines of the file:
----------------------------------------
25/07/18	23:40:00	3,500	21,965	44,910	1942,625	3,490	21,750	45,115	2122,750	0,000	0,000	0,000	0,000
26/07/18	00:00:00	3,500	21,695	45,220	1940,625	3,470	21,555	45,155	2122,125	3,480	20,870	50,095	2077,125
26/07/18	00:20:00	0,000	0,000	0,000	0,000	3,470	21,365	45,040	2121,000	3,490	20,700	48,595	2076,625
26/07/18	00:40:00	3,490	21,090	45,005	1936,125	3,480	21,150	44,500	2120,875	3,480	20,510	49,005	2075,375
26/07/18	01:00:00	3,500	20,760	45,405	1934,625	3,470	20,980	45,195	2120,375	3,480	20,450	49,120	2075,375
----------------------------------------
Use this information to set 'sep' and 'column_names' in load_sensor_directory().



## Step 2 · Load and Merge Raw Sensor Files
Load all pieces of the sensor data.

In [7]:
# === USER INPUT ===
RAW_FOLDER    = 'data/raw/sensor'
FILE_EXT      = '.adc'
SEPARATOR     = '\t'
HEADER        = None

# Map each station's sequential fields to the standard pipeline schema.
# Use None if a sensor reading is missing in your raw data.
STATIONS = {
    'st01': ['charge', 'temp', 'hum', 'absinc'],
    'st02': ['charge', 'temp', 'hum', 'absinc'],
    'st03': ['charge', 'temp', 'hum', 'absinc']
}
# ==================

df_raw = load_sensor_directory(
    folder_path=RAW_FOLDER,
    extension=FILE_EXT,
    sep=SEPARATOR,
    header=HEADER,
    column_names=None,
    date_col=0,  # Map 1st column to date
    time_col=1,  # Map 2nd column to time
    save_combined=False
)

print(f"\nLoaded dataset shape: {df_raw.shape}")
print(f"Date range: {df_raw.index.min()} → {df_raw.index.max()}")

# Organize data into independent station datasets using the dataloader module
stations_dict = organize_sensor_data(df_raw, STATIONS)
print(f"Organized {len(stations_dict)} independent stations from raw data.")

Found 1911 files matching '.adc'. Processing...


Loading sensor data:   0%|          | 0/1911 [00:00<?, ?it/s]d:\Jupyter\neuralprophet\heritageshm\dataloader.py:71: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['datetime'] = pd.to_datetime(df[date_col].astype(str) + ' ' + df[time_col].astype(str), dayfirst=True, errors='coerce')
d:\Jupyter\neuralprophet\heritageshm\dataloader.py:71: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['datetime'] = pd.to_datetime(df[date_col].astype(str) + ' ' + df[time_col].astype(str), dayfirst=True, errors='coerce')
d:\Jupyter\neuralprophet\heritageshm\dataloader.py:71: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expecte


Loaded dataset shape: (132590, 12)
Date range: 2018-07-25 23:40:00 → 2025-02-05 12:20:00
Organized 3 independent stations from raw data.


## Step 3 · Signal Cleaning and Saving
Clean the data and save to the interim format as a standardized CSV series.

In [9]:
# Iterate through all the datasets created for the clean process
from IPython.display import display
import os

os.makedirs('data/interim/sensor', exist_ok=True)

for st, df_st in stations_dict.items():
    print(f"--- Processing station {st} ---")
    
    # 1. Clean signal: power loss and outliers
    outlier_thresholds = {'absinc': -500}
    df_clean = clean_signal(df_st, valid_charge_col='charge', outlier_thresholds=outlier_thresholds)
    
    # 2. Apply temperature calibration/compensation
    if len(df_clean) > 0 and 'temp' in df_clean.columns and 'absinc' in df_clean.columns:
        df_clean = apply_compensation(
            df=df_clean, 
            target_col='absinc', 
            new_col_name='absinc_clean', 
            comp_func=temp_compensation, 
            normalize=True, 
            temp_col='temp', 
            comp_coeff=0.005
        )
        # Keep only the compensated absinc (drop raw, rename compensated)
        df_clean = df_clean.drop(columns=['absinc'])
        df_clean = df_clean.rename(columns={'absinc_clean': 'absinc'})
    
    # 3. Save to a specific file
    output_path = f"data/interim/sensor/{st}_preprocessed.csv"
    df_clean.to_csv(output_path)
    print(f"Preprocessed {st} data saved to: {output_path}")

    # 4. Preview the cleaned dataset
    print(f"Shape: {df_clean.shape} | Date range: {df_clean.index.min()} → {df_clean.index.max()}")
    display(df_clean.head())
    print()


--- Processing station st01 ---
Dropped 48136 rows due to zero charge in 'charge'.
Preprocessed st01 data saved to: data/interim/sensor/st01_preprocessed.csv
Shape: (84454, 4) | Date range: 2018-07-25 23:40:00 → 2022-09-22 17:40:00


,charge,temp,hum,absinc
datetime,,,,
2018-07-25 23:40:00,3.50,21.965,44.910,0.000
2018-07-26 00:00:00,3.50,21.695,45.220,-0.650
2018-07-26 00:40:00,3.49,21.090,45.005,-2.125
2018-07-26 01:00:00,3.50,20.760,45.405,-1.975
2018-07-26 01:20:00,3.50,20.900,46.130,-4.050



--- Processing station st02 ---
Dropped 16028 rows due to zero charge in 'charge'.
Preprocessed st02 data saved to: data/interim/sensor/st02_preprocessed.csv
Shape: (116562, 4) | Date range: 2018-07-25 23:40:00 → 2025-02-05 12:20:00


,charge,temp,hum,absinc
datetime,,,,
2018-07-25 23:40:00,3.49,21.750,45.115,0.000
2018-07-26 00:00:00,3.47,21.555,45.155,0.350
2018-07-26 00:20:00,3.47,21.365,45.040,0.175
2018-07-26 00:40:00,3.48,21.150,44.500,1.125
2018-07-26 01:00:00,3.47,20.980,45.195,1.475



--- Processing station st03 ---
Dropped 62199 rows due to zero charge in 'charge'.
Preprocessed st03 data saved to: data/interim/sensor/st03_preprocessed.csv
Shape: (70391, 4) | Date range: 2018-07-26 00:00:00 → 2023-06-24 16:00:00


,charge,temp,hum,absinc
datetime,,,,
2018-07-26 00:00:00,3.48,20.87,50.095,0.000
2018-07-26 00:20:00,3.49,20.70,48.595,0.350
2018-07-26 00:40:00,3.48,20.51,49.005,0.050
2018-07-26 01:00:00,3.48,20.45,49.120,0.350
2018-07-26 01:20:00,3.48,20.14,50.200,-0.475
